# 2. 线性神经网络

## 1. 线性回归

线性回归是机器学习中最基础的模型之一。给定 $d$ 个特征 $\mathbf{x} = [x_1, x_2, \dots, x_d]^\top$，线性回归的预测值为：

$$\hat{y} = \mathbf{w}^\top \mathbf{x} + b$$

其中 $\mathbf{w}$ 是权重向量，$b$ 是偏置标量。

### 损失函数（均方误差 MSE）

使用均方误差来衡量预测值与真实值之间的差距：

$$\ell(\mathbf{w}, b) = \frac{1}{2} (\hat{y} - y)^2$$

### 梯度下降

通过梯度下降来最小化损失函数。每一步沿着损失函数梯度的反方向更新参数：

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \frac{\partial \ell}{\partial \mathbf{w}}$$
$$b \leftarrow b - \eta \frac{\partial \ell}{\partial b}$$

In [ ]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

## 2. 线性回归的从零开始实现

### 生成合成数据集

In [2]:
def synthetic_data(w, b, num_examples):
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

In [3]:
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.scatter(features[:, 0], labels, alpha=0.5)
plt.xlabel('x1')
plt.ylabel('y')
plt.subplot(1, 2, 2)
plt.scatter(features[:, 1], labels, alpha=0.5)
plt.xlabel('x2')
plt.ylabel('y')
plt.show()

### 读取数据集（小批量）

In [4]:
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(indices[i: min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

import random
batch_size = 10
for X, y in data_iter(batch_size, features, labels):
    print(X.shape, y.shape)
    break

### 初始化参数、定义模型、损失函数和优化算法

In [5]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

def linreg(X, w, b):
    return torch.matmul(X, w) + b

def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

### 训练循环

In [6]:
lr = 0.03
num_epochs = 3
net = linreg
loss = squared_loss

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)
        l.sum().backward()
        sgd([w, b], lr, batch_size)
    with torch.no_grad():
        train_loss = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_loss.mean()):.6f}')

In [7]:
print(f'w的估计误差: {true_w - w.reshape(true_w.shape)}')
print(f'b的估计误差: {true_b - b}')

## 3. 线性回归的简洁实现

### 生成数据集

In [8]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

### 使用 DataLoader 读取数据

In [9]:
dataset = TensorDataset(features, labels)
data_iter = DataLoader(dataset, batch_size=10, shuffle=True)

for X, y in data_iter:
    print(X.shape, y.shape)
    break

### 定义模型、损失函数和优化器

In [10]:
net = nn.Sequential(nn.Linear(2, 1))
loss = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.03)

### 训练循环

In [11]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l.item():.6f}')

In [12]:
w = net[0].weight.data
b = net[0].bias.data
print(f'w的估计误差: {true_w - w.reshape(true_w.shape)}')
print(f'b的估计误差: {true_b - b}')

## 4. Softmax 回归

Softmax 回归用于多分类问题。对于 $k$ 个类别，给定输入 $\mathbf{x}$，输出属于每个类别的概率：

$$\hat{\mathbf{y}} = \text{softmax}(\mathbf{o}) \quad \text{其中} \quad o_k = \mathbf{w}_k^\top \mathbf{x} + b_k$$

Softmax函数将输出转换为概率分布：

$$\hat{y}_k = \frac{\exp(o_k)}{\sum_{j=1}^k \exp(o_j)}$$

### 交叉熵损失

交叉熵损失衡量真实标签分布与预测分布之间的差异：

$$\ell(\mathbf{y}, \hat{\mathbf{y}}) = -\sum_{j=1}^k y_j \log \hat{y}_j = -\log \hat{y}_y$$

## 5. Softmax 回归的从零开始实现

### 加载 Fashion-MNIST 数据集

In [13]:
batch_size = 256

mnist_train = torchvision.datasets.FashionMNIST(
    root='./data', train=True, transform=torchvision.transforms.ToTensor(),
    download=True)
mnist_test = torchvision.datasets.FashionMNIST(
    root='./data', train=False, transform=torchvision.transforms.ToTensor(),
    download=True)

train_iter = DataLoader(mnist_train, batch_size, shuffle=True)
test_iter = DataLoader(mnist_test, batch_size, shuffle=False)

In [14]:
def get_fashion_mnist_labels(labels):
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
                   'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [text_labels[int(i)] for i in labels]

X, y = next(iter(train_iter))
_, axes = plt.subplots(1, 5, figsize=(8, 4))
for i in range(5):
    axes[i].imshow(X[i][0], cmap='gray')
    axes[i].set_title(get_fashion_mnist_labels([y[i]])[0])
    axes[i].axis('off')
plt.show()

### 初始化参数、定义 softmax 和交叉熵

In [15]:
num_inputs = 784
num_outputs = 10

W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)

def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition

def net(X):
    return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)

def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])

def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

### 训练

In [16]:
def evaluate_accuracy(net, data_iter):
    metric = [0.0, 0.0]  # correct count, total count
    with torch.no_grad():
        for X, y in data_iter:
            metric[0] += accuracy(net(X), y)
            metric[1] += y.numel()
    return metric[0] / metric[1]

num_epochs = 5
lr = 0.1

for epoch in range(num_epochs):
    metric = [0.0, 0.0]  # total loss, total count
    for X, y in train_iter:
        y_hat = net(X)
        l = cross_entropy(y_hat, y)
        l.sum().backward()
        sgd([W, b], lr, batch_size)
        metric[0] += float(l.sum())
        metric[1] += y.numel()
    train_loss = metric[0] / metric[1]
    test_acc = evaluate_accuracy(net, test_iter)
    print(f'epoch {epoch + 1}, loss {train_loss:.6f}, test acc {test_acc:.4f}')

### 预测

In [17]:
X, y = next(iter(test_iter))
preds = net(X).argmax(axis=1)
_, axes = plt.subplots(1, 6, figsize=(10, 4))
for i in range(6):
    axes[i].imshow(X[i][0], cmap='gray')
    axes[i].set_title(f'{get_fashion_mnist_labels([preds[i]])[0]}\n(true: {get_fashion_mnist_labels([y[i]])[0]})')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 6. Softmax 回归的简洁实现

In [18]:
train_iter = DataLoader(mnist_train, batch_size, shuffle=True)
test_iter = DataLoader(mnist_test, batch_size, shuffle=False)

net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights)

loss = nn.CrossEntropyLoss(reduction='none')
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)

In [19]:
num_epochs = 5
for epoch in range(num_epochs):
    metric = [0.0, 0.0]
    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y)
        optimizer.zero_grad()
        l.sum().backward()
        optimizer.step()
        metric[0] += float(l.sum())
        metric[1] += y.numel()
    train_loss = metric[0] / metric[1]
    test_acc = evaluate_accuracy(net, test_iter)
    print(f'epoch {epoch + 1}, loss {train_loss:.6f}, test acc {test_acc:.4f}')

In [20]:
X, y = next(iter(test_iter))
preds = net(X).argmax(axis=1)
_, axes = plt.subplots(1, 6, figsize=(10, 4))
for i in range(6):
    axes[i].imshow(X[i][0], cmap='gray')
    axes[i].set_title(f'{get_fashion_mnist_labels([preds[i]])[0]}\n(true: {get_fashion_mnist_labels([y[i]])[0]})')
    axes[i].axis('off')
plt.tight_layout()
plt.show()